> **Not part of the delivered chain.** This notebook documents the construction and the variants that were screened, and writes the earlier v1.2 schema. The delivered series is built by `pipeline/build_series.py`, which produces the v3.1_RAS schema that step 20 consumes.

# Part 3 — Intra-US MRIOT, v1.2-A construction (margins via the md0 chain)

Clean notebook that uses **only the v3 functions** (`reconstruct_bilateral_3`, `build_margin_tensors`, `compute_use_shares_3`, `build_Z_3`, `build_F_3`). Margins (dm0 local, nm0 inter-state) are routed to whoever buys the goods that carry them (`good -> margin type (nm0) -> carrier good (md0) -> buyer`). Validated on 2017: row identity exact (0.000%), column identity residual ~0.94%.

Structure: **Setup → Functions → Construction → Diagnostic → Analyse**.

## Setup

#### Installations

In [ ]:
from paths import ROOT
import sys
!{sys.executable} -m pip install gdx2py

In [ ]:
import sys
!{sys.executable} -m pip install gamspy-base

#### Imports

In [ ]:
from gdx2py import GdxFile
import gamspy_base
import os
import pandas as pd
from gdx2py.gams import GAMSParameter, GAMSSet
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import product

In [ ]:
gamspybase_directory = gamspy_base.directory
print(gamspybase_directory)

In [ ]:
ROOT

In [ ]:
path_windc_gdx = str(ROOT / "data/raw/GTAPWiNDC/data/core/WiNDCdatabase.gdx")

In [ ]:
gdx = GdxFile(path_windc_gdx, gams_dir=gamspybase_directory)
print(list(gdx))

In [ ]:
# ── 1. Load all parameters ───────────────────────────────────────────────────
params = {}
for name, obj in gdx:
    if isinstance(obj, GAMSParameter):
        s = obj.to_pandas()
        if s is not None and len(s) > 0:
            df = s.reset_index()
            df.columns = list(df.columns[:-1]) + ['value']
            params[name] = df
# params contains all years and regions. We will filter it later when we need to build the IOT for a specific year and region.

In [ ]:
# Regions: union of all states present in xn0_ or nd0_
all_xn0 = set(params['xn0_']['r'].unique())
all_nd0 = set(params['nd0_']['r'].unique())
regions = sorted(all_xn0 | all_nd0)
n = len(regions)
print(n)

In [ ]:
# Economic (GDP-weighted) centroids -- the delivered reference points of the gravity
# distance matrix. Built by 02_economic_centroids.ipynb (BEA CAGDP2 county GDP + the
# 2020 Census county population centroids). Loaded here as {abbr: (lat, lon)}.
_cen = pd.read_csv(ROOT / "data/interim/economic_centroids.csv")
COORDS = {r.abbr: (r.lat, r.lon) for r in _cen.itertuples(index=False)}


In [ ]:
# distance function
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi, dlam = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# Distance matrix between all pairs of regions, great-circle (haversine) between the
# GDP-weighted ECONOMIC CENTROIDS of each region (data/interim/economic_centroids.csv,
# built in 02_economic_centroids.ipynb). These are the delivered reference points; the
# earlier capital-based prior and the alternative (geometric, population-weighted)
# centroids are compared in analysis/distance_variants.py of the source project.
D= pd.DataFrame(index=regions, columns=regions, dtype=float)
for i, j in product(regions, regions):
    D.loc[i, j] = np.nan if i == j else haversine(*COORDS[i], *COORDS[j])

D_np = D.values.copy()

missing = [r for r in regions if r not in COORDS]
if missing:
    print(f"Warning: regions without coordinates: {missing}")
print(f"{len(regions)} regions | distance range: "
      f"{D_np[~np.isnan(D_np)].min():.0f}-{D_np[~np.isnan(D_np)].max():.0f} km")


In [ ]:
# Example: load use matrix for New York 2017 as a numpy array (goods × sectors)
def load_matrix(param_name, year, regions, sectors):
    df = params[param_name]
    dim = 'g' if 'g' in df.columns else 's'
    return (df[df['yr'] == year]
            .groupby(['r', dim])['value'].sum()
            .unstack(dim)
            .reindex(index=regions, columns=sectors, fill_value=0.0)
            .fillna(0.0)          
            .values)

In [ ]:
def load_year_data(year, regions, sectors, names_params):
    """Load all IO matrices for a given year. Returns a dict of arrays."""
    n, S = len(regions), len(sectors)

    names = names_params
    mats = {name: load_matrix(name, year, regions, sectors) for name in names}

    absorption = mats['dd0_'] + mats['nd0_'] + mats['m0_']
    absorption_safe = np.where(absorption < 1e-10, 1.0, absorption)

    id0_tensor = (params['id0_'][params['id0_']['yr'] == year]
                  .groupby(['r', 'g', 's'])['value'].sum()
                  .unstack('s')
                  .reindex(pd.MultiIndex.from_product([regions, sectors], names=['r', 'g']),
                           fill_value=0.0)
                  .reindex(columns=sectors, fill_value=0.0)
                  .fillna(0.0)
                  .values
                  .reshape(n, S, S))

    return {**mats, 'absorption': absorption, 'absorption_safe': absorption_safe,
            'id0': id0_tensor}

In [ ]:
EXCLUDED = {}
sectors = sorted(s for s in params['xn0_']['g'].unique() if s not in EXCLUDED)

In [ ]:
# Index maps and dimensions
region_to_idx = {r: i for i, r in enumerate(regions)}
sector_to_idx = {s: i for i, s in enumerate(sectors)}
n, S = len(regions), len(sectors)
print(f'{n} regions x {S} sectors')

## Reconstruction of bilateral trades

This code takes the state SRIOs (WiNDC base, where inter-state trade exists
only in aggregated form via a *national pool*) and **reconstructs the bilateral
state→state flows** from the imports/exports to the national pool and a simple
gravity model:

$$T_{(s,i),j} = X_{(s,i)} \cdot M_{(s,j)} \cdot d_{ij}^{-\gamma}$$

where $X_{(s,i)}$ is the export of state $i$ to the pool for good $s$ (`xn0`), $M_{(s,j)}$ the demand of state $j$ to the pool (`nd0` for direct trade, `nm0` summed over margin types
for margins) and $d_{ij}$ the distance between capitals (haversine). The
gravity seed is then calibrated by **RAS** to exactly restore the WiNDC marginals:
$\sum_j T_{(s,i),j} = $ `xn0`, $\sum_i T_{(s,i),j} = $ `nd0` (or `nm0`) — so
that all the balance identities verified on the national base stay true
after bilateralization. Direct trade (`nd0`) and margins (`nm0`) are reconstructed
separately but draw from the same export pool `xn0`.

Then the bilateral flows from a state $i$ to a state $j$ for sector $s$ are
**distributed according to the absorption structure from the national pool**: for each
delivered good, the use shares (`compute_use_shares_3`) split the flow between
buying sectors (Z) and final demand C/I/G (F). The margins (dm0 local, nm0
inter-state) are routed to whoever buys the goods that carry them, via the chain
`good → margin type (nm0) → carrier good (md0) → buyer`.

The notebook uses **only the v3 functions** (`reconstruct_bilateral_3`,
`build_margin_tensors`, `compute_use_shares_3`, `build_Z_3`, `build_F_3`).

### Functions

In [ ]:
def ras_robust(seed, X, M, max_iter=2000, tol=1e-8):
    """RAS with convergence tracking. RAS lets us preserve the table's initial aggregate structure at the start.
    Indeed the approximation of the formula T = X.M.D^gamma distorts the matrix and does not guarantee that the resulting table
    is consistent with the aggregate flows observed at the start,
    i.e. that the total of what leaves as good g from state i toward states j equals the exports of state i for good g
    toward the NP in the initial table (row agreement);
    and that the total of what enters as good g into state j from states i equals the imports of state j for good g
    from the NP in the initial table (column agreement).

    The RAS algorithm proportionally scales up a whole row and a whole column at each iteration,
    until the row and column totals are close enough to the target totals (X and M).

    Args:
    seed: starting matrix (nxn)
    X: vector of row totals (n,)
    M: vector of column totals (n,)
    max_iter: maximum number of iterations
    tol: convergence tolerance

    Returns:
        T: adjusted matrix
        converged: boolean indicating whether convergence was reached for each sector
        final_err: final error (max of the deviations from the totals)
        iters: number of iterations performed
        initial_err: initial error (before adjustment)
    """
    T = seed.copy().astype(float) #starting nxn matrix created from the gravity seed with the chosen gamma
    r = X.values if hasattr(X, 'values') else X
    c = M.values if hasattr(M, 'values') else M
    initial_err = max(np.abs(T.sum(axis=1) - r).max(),
                      np.abs(T.sum(axis=0) - c).max())
    for it in range(max_iter):
        rs = T.sum(axis=1); rs[rs == 0] = 1
        T *= (r / rs)[:, None]
        cs = T.sum(axis=0); cs[cs == 0] = 1
        T *= (c / cs)[None, :]
        err = max(np.abs(T.sum(axis=1) - r).max(),
                  np.abs(T.sum(axis=0) - c).max())
        if err < tol:
            return T, True, err, it + 1, initial_err
    return T, False, err, max_iter, initial_err

In [ ]:
def reconstruct_bilateral_3(xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions,
                            D_np, gamma_trade=1.0, gamma_margin=1.0,
                            imbalance_skip=1e-4):
    """
    Reconstruct bilateral matrices separately for trade links (nd0) and margin
    links (nm0), each with its own spatial distribution, via gravity + RAS.

    Both flows draw from the same export pool xn0 (row/origin marginal) but have
    distinct absorption targets (column/destination marginals): nd0 for direct
    trade, sum_m nm0 for margin absorption. Per commodity, the export pool is
    split in proportion to the national total absorbed through each channel, so
    that each layer carries equal row and column totals:

        sum_r X_trade(r)  == sum_r nd0(r)        (per commodity)
        sum_r X_margin(r) == sum_r nm0(r)        (per commodity)

    That equality is the FEASIBILITY condition of the RAS, not a property of the
    gravity seed: the raw product X.M.d^-gamma reproduces neither marginal. It is
    ras_robust that restores them, so that after fitting

        row_sums(T_trade) + row_sums(T_margin) = xn0        (per region)
        col_sums(T_trade)                      = nd0        (per region)
        col_sums(T_margin)                     = sum_m nm0  (per region)

    to the fitting tolerance. The row identity holds only when total_M == total_X,
    since the split rescales the pool by total_M / total_X; the WiNDC identity
    sum_r xn0 = sum_r (nd0 + sum_m nm0) makes that exact per commodity, and the
    imbalance guard below is what catches a source where it is not.

    Note that RAS absorbs any monotone rescaling of the row and column masses into
    its own diagonal scalings, so exponents on X and M would leave T unchanged:
    the only parameter of the seed that survives the fit is gamma.

    Returns:
        T_trade_all  (dict sector -> n x n array)
        T_margin_all (dict sector -> n x n array)
        df_log       (convergence log).
    """
    n = len(regions)

    with np.errstate(divide='ignore', invalid='ignore'):
        friction_trade  = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma_trade))
        friction_margin = np.where(np.isnan(D_np), 0.0, D_np ** (-gamma_margin))

    def _ras_one_flow(X_row, M_col, friction):
        """Run a single gravity+RAS reconstruction; X_row/M_col are pd.Series."""
        tX, tM = X_row.sum(), M_col.sum()
        if tX < 1e-10 or tM < 1e-10:           # flow absent -> empty matrix
            return np.zeros((n, n)), True, 0.0, 0, 0.0
        # No epsilon floor is added to the seed. It would be redundant: friction is
        # d^-gamma with d finite and strictly positive off the diagonal, so the seed
        # has no structural zero for RAS to trip on, and a zero row/column of X or M
        # carries a zero target anyway. Measured on 2017, adding the former
        # 1e-8 * outer(X/tX, M/tM) term left the iteration count identical on all 79
        # (commodity, layer) fits and only perturbed the result, because
        # outer(X,M)*f + 1e-8*outer(X/tX,M/tM) == outer(X,M) * (f + 1e-8/(tX*tM)),
        # i.e. it was a scale-DEPENDENT additive perturbation of the friction: 5e-9 of
        # the smallest friction on a median commodity but 1.1% of it on pipeline
        # transport, the smallest pool, shifting that layer by 5.1e-4 in relative L1.
        seed = np.outer(X_row.values, M_col.values) * friction
        np.fill_diagonal(seed, 0.0)
        return ras_robust(seed, X_row, M_col)

    def _status(converged, err, tol_soft=1e-6):
        if converged:        return 'ok'
        if err < tol_soft:   return 'ok_soft'
        return 'FAILED'

    T_trade_all, T_margin_all = {}, {}
    log_ras = []

    for g in sectors:
        g_i = sector_to_idx[g]
        X_g  = pd.Series(xn0_mat[:, g_i], index=regions)
        nd_g = pd.Series(nd0_mat[:, g_i], index=regions)
        nm_g = pd.Series(nm0_mat[:, g_i], index=regions)

        total_X  = X_g.sum()
        total_nd = nd_g.sum()
        total_nm = nm_g.sum()
        total_M  = total_nd + total_nm

        # Two distinct cases, which the previous single test conflated.
        # (a) The commodity has no national pool activity at all: nothing to
        #     reconstruct, an empty layer pair is the correct answer.
        if total_X < 1e-10 or total_M < 1e-10:
            T_trade_all[g]  = np.zeros((n, n))
            T_margin_all[g] = np.zeros((n, n))
            log_ras.append({'sector': g, 'status': 'no_pool_activity',
                            'err': 0.0, 'iters': 0})
            continue

        # (b) The marginals disagree. WiNDC closes the pool per commodity,
        #     sum_r xn0 == sum_r (nd0 + sum_m nm0), so any disagreement beyond float
        #     accumulation means the inputs are misaligned (wrong year, wrong margin
        #     axis, a source vintage that broke the identity) and the fit below would
        #     silently return a rescaled table. Measured over 1997-2023 x 71
        #     commodities the worst residual is 2.5e-7 (pipeline transport, 2005) and
        #     nothing exceeds 1e-6, whereas dropping nm0 from the marginal - the
        #     documented failure mode - puts 8 commodities above 0.5. The threshold
        #     sits between the two, and this is now an error rather than a silent
        #     zero-fill: a dropped commodity used to disappear from the table with no
        #     trace outside the log.
        imbalance = abs(total_X - total_M) / total_X
        if imbalance > imbalance_skip:
            raise ValueError(
                f"national pool marginals disagree for commodity {g!r}: "
                f"sum xn0 = {total_X:.6g}, sum (nd0 + nm0) = {total_M:.6g}, "
                f"relative imbalance {imbalance:.3e} > {imbalance_skip:.0e}. "
                f"The WiNDC pool identity should close this to ~1e-7; check that "
                f"nm0 is summed over margin types and that all inputs are the same "
                f"year.")

        # Split each region's export pool between trade and margin proportionally
        # to their (rescaled) national totals -> each flow is self-balanced.
        X_trade  = X_g * (total_nd / total_X)
        X_margin = X_g * (total_nm / total_X)

        T_trade,  c_t, err_t, it_t, _ = _ras_one_flow(X_trade,  nd_g, friction_trade)
        T_margin, c_m, err_m, it_m, _ = _ras_one_flow(X_margin, nm_g, friction_margin)

        T_trade_all[g]  = T_trade
        T_margin_all[g] = T_margin

        log_ras.append({'sector': g,
                        'status_trade':  _status(c_t, err_t),
                        'status_margin': _status(c_m, err_m),
                        'err_trade': err_t, 'err_margin': err_m,
                        'iters_trade': it_t, 'iters_margin': it_m,
                        'nd_total': total_nd, 'nm_total': total_nm,
                        'seed': 'gravity'})

    return T_trade_all, T_margin_all, pd.DataFrame(log_ras)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# v1.2 margin routing — the two helpers consumed by build_Z_3 / build_F_3.
#   build_margin_tensors : load nm0 (r,g,m) and md0 (r,m,g) as dense tensors.
#   compute_use_shares_3 : DIRECT shares (as v2) + MARGIN shares routed via md0.
# ──────────────────────────────────────────────────────────────────────────────
def build_margin_tensors(year, regions, sectors, margins=('trd', 'trn')):
    """
    Build the margin tensors used to route trade/transport margins through Z and F.

    Returns
    -------
    nm0_rgm : (n, S, M)  national margin SUPPLIED by good g, of margin type m
    md0_rmg : (n, M, S)  margin DEMAND of type m in the absorption of good g
    m_set   : list of margin types m
    """
    n, S, M = len(regions), len(sectors), len(margins)
    margins = list(margins)

    nm0_rgm = (params['nm0_'][params['nm0_']['yr'] == year]
               .groupby(['r', 'g', 'm'])['value'].sum()
               .reindex(pd.MultiIndex.from_product([regions, sectors, margins],
                                                   names=['r', 'g', 'm']), fill_value=0.0)
               .values.reshape(n, S, M))

    md0_rmg = (params['md0_'][params['md0_']['yr'] == year]
               .groupby(['r', 'm', 'g'])['value'].sum()
               .reindex(pd.MultiIndex.from_product([regions, margins, sectors],
                                                   names=['r', 'm', 'g']), fill_value=0.0)
               .values.reshape(n, M, S))

    return nm0_rgm, md0_rmg, margins


def compute_use_shares_3(id0_df, cd0_mat, i0_mat, g0_mat, nm0_rgm, md0_rmg):
    """
    Use shares for the v1.2 build, split into a DIRECT and a MARGIN channel.
    Each channel sums to 1 over {buying sectors s} + {C, I, G} for every (r, g),
    so build_Z_3 / build_F_3 conserve dd0, dm0 and the bilateral T flows exactly.

    DIRECT  (d_int, dC, dI, dG): purchaser-price absorption structure of good g
        (id0/cd0/i0/g0 over total demand), with a fallback to the region's average
        input profile for goods that record no demand of their own.

    MARGIN  (m_int, mC, mI, mG): a unit of margin good g is a markup attached to the
        delivery of OTHER goods, so it is routed along the chain
            g --(nm0)--> margin type m --(md0)--> carrier good g' --(direct shares)--> buyer
        i.e. the margin lands on whoever buys the goods that carry it.
    """
    n, S = id0_df.shape[0], id0_df.shape[1]

    # DIRECT channel (identical to compute_use_shares_2)
    total_demand = id0_df.sum(axis=2) + cd0_mat + i0_mat + g0_mat
    safe = np.where(total_demand < 1e-10, 1.0, total_demand)
    d_int = id0_df / safe[:, :, None]
    dC, dI, dG = cd0_mat / safe, i0_mat / safe, g0_mat / safe

    # fallback: goods with no recorded demand (margin goods) -> avg input profile
    total_inputs = id0_df.sum(axis=1)
    row_sum = total_inputs.sum(axis=1, keepdims=True)
    fallback = total_inputs / np.where(row_sum < 1e-10, 1.0, row_sum)
    mask_zero = (total_demand < 1e-10)[:, :, None]
    d_int = np.where(mask_zero, fallback[:, None, :], d_int)

    # MARGIN channel (md0 chain)
    # margin-type mix supplied by good g (share over m). nm0-only == nm0+dm0 here.
    w = nm0_rgm.astype(float)
    w_sum = w.sum(axis=2, keepdims=True)
    w = np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum > 1e-12)

    # buyer profile of each margin type m: who absorbs the goods that carry it
    tot_md = md0_rmg.sum(axis=2)                          # (n, M)
    safe_md = np.where(tot_md < 1e-12, 1.0, tot_md)
    bf_int = np.einsum('rmh,rhs->rms', md0_rmg, d_int) / safe_md[:, :, None]
    bf_C = (md0_rmg * dC[:, None, :]).sum(axis=2) / safe_md
    bf_I = (md0_rmg * dI[:, None, :]).sum(axis=2) / safe_md
    bf_G = (md0_rmg * dG[:, None, :]).sum(axis=2) / safe_md

    # weight the buyer profiles by good g's margin-type mix
    m_int = np.einsum('rgm,rms->rgs', w, bf_int)
    mC = np.einsum('rgm,rm->rg', w, bf_C)
    mI = np.einsum('rgm,rm->rg', w, bf_I)
    mG = np.einsum('rgm,rm->rg', w, bf_G)

    return d_int, dC, dI, dG, m_int, mC, mI, mG


In [ ]:
def build_Z_3(dd0_mat, dm0_mat, T_dir, T_mar,
              ush_dir, ush_mar, sectors, n, S):
    Z = np.zeros((n, S, n, S))
    for r in range(n):                                  # local: dd0 direct + dm0 margin
        Z[r, :, r, :] += dd0_mat[r, :, None] * ush_dir[r, :, :]
        Z[r, :, r, :] += dm0_mat[r, :, None] * ush_mar[r, :, :]
    for gi, g in enumerate(sectors):                    # interstate flows
        Z[:, gi, :, :] += T_dir[g][:, :, None] * ush_dir[:, gi, :][None, :, :]
        Z[:, gi, :, :] += T_mar[g][:, :, None] * ush_mar[:, gi, :][None, :, :]
    return Z.reshape(n * S, n * S)


def build_F_3(dd0_mat, dm0_mat, T_dir, T_mar,
              dC, dI, dG, mC, mI, mG, sectors, n, S):
    F = np.zeros((n, S, n, 3))
    for r in range(n):
        F[r, :, r, 0] += dd0_mat[r, :] * dC[r, :] + dm0_mat[r, :] * mC[r, :]
        F[r, :, r, 1] += dd0_mat[r, :] * dI[r, :] + dm0_mat[r, :] * mI[r, :]
        F[r, :, r, 2] += dd0_mat[r, :] * dG[r, :] + dm0_mat[r, :] * mG[r, :]
    for gi, g in enumerate(sectors):
        F[:, gi, :, 0] += T_dir[g] * dC[:, gi][None, :] + T_mar[g] * mC[:, gi][None, :]
        F[:, gi, :, 1] += T_dir[g] * dI[:, gi][None, :] + T_mar[g] * mI[:, gi][None, :]
        F[:, gi, :, 2] += T_dir[g] * dG[:, gi][None, :] + T_mar[g] * mG[:, gi][None, :]
    return F.reshape(n * S, n * 3)


### Construction

In [ ]:
# ── Build configuration ─────────────────────────────────────────────────
# YEARS accepts a single year ('2017') OR a list (['2015', '2016', '2017']).
# PRODUCTION DEFAULT = ALL years, so a run-all regenerates the complete,
# schema-homogeneous grav_fric_v3.1 AND grav_fric_v3.1_RAS builds
# (the committed single-year default previously left grav_fric_v3.1_RAS
# incomplete: only 2017 had the full 18-key schema + VA convention).
# VERSION tags the output directory + filenames so successive experiments do not
# overwrite each other: change it (e.g. 'v3', 'v3_gamma2', 'v3_test') for a fresh
# build location instead of overwriting grav_fric_v3.1 in place.
from pathlib import Path

YEARS   = ['1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005',
           '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014',
           '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023']
# YEARS = '2017'           # single-year test switch
VERSION = 'v3.1'             # free-form tag -> grav_fric_<VERSION>
GAMMA_TRADE  = 1.0         # gravity friction exponent for direct trade (nd0)
GAMMA_MARGIN = 1.0         # gravity friction exponent for margin flows (nm0)
COMMENT = ''               # free-text note on this build -> written to README.md

years = [YEARS] if isinstance(YEARS, str) else list(YEARS)

names = ['dd0_', 'nd0_', 'xn0_', 'xd0_', 'x0_', 'm0_',
         'cd0_', 'i0_', 'g0_', 'ld0_', 'kd0_', 'ty0_']

# Base output root + version-aware path helpers (no hard-coded OUT_V3).
IOT_ROOT = ROOT / "data/interim/IOT/IOT_USA"

def build_dir(version):
    """Output directory grav_fric_<version> (created on demand)."""
    d = IOT_ROOT / f'grav_fric_{version}'
    d.mkdir(parents=True, exist_ok=True)
    return d

def build_path(version, year):
    """Full npz path for one (version, year)."""
    return build_dir(version) / f'IOT_{year}.npz'

print('config:', years, '| version =', VERSION)


In [ ]:
def build_table(year, gamma_trade=GAMMA_TRADE, gamma_margin=GAMMA_MARGIN):
    """Build the intra-US MRIOT for a single year.

    Returns a dict of arrays ready for np.savez_compressed (grav_fric schema).
    No global side effects: callable in a loop over several years.
    Schema note: on top of the original v3 blocks, the useful v2-schema
    bookkeeping blocks are saved too (Y = ys0 ground truth, M_fd_C/I/G = final-
    demand split of ROW imports with the direct shares), so v3 is a strict
    superset of the v1.2 schema for diagnostics.
    """
    data = load_year_data(year, regions, sectors, names)
    dd0_mat = data['dd0_']; nd0_mat = data['nd0_']
    xn0_mat = data['xn0_']; xd0_mat = data['xd0_']; x0_mat = data['x0_']
    m0_mat  = data['m0_'];  cd0_mat = data['cd0_']
    i0_mat  = data['i0_'];  g0_mat  = data['g0_']
    ld0_mat = data['ld0_']; kd0_mat = data['kd0_']
    id0_df  = data['id0']

    # national margin supply aligned (r, g), summed over margin types
    nm0_mat = (params['nm0_'][params['nm0_']['yr'] == year]
               .groupby(['r', 'g'])['value'].sum().unstack('g')
               .reindex(index=regions, columns=sectors, fill_value=0.0).fillna(0.0).values)

    # bilateral reconstruction (v3): separate trade (nd0) + margin (nm0), shared pool xn0
    T_dir, T_mar, log = reconstruct_bilateral_3(
        xn0_mat, nd0_mat, nm0_mat, sectors, sector_to_idx, regions, D_np,
        gamma_trade=gamma_trade, gamma_margin=gamma_margin)

    # use shares (DIRECT + MARGIN via the md0 chain) and assembly
    dm0_mat = xd0_mat - dd0_mat
    nm0_rgm, md0_rmg, m_set = build_margin_tensors(year, regions, sectors)
    d_int, dC, dI, dG, m_int, mC, mI, mG = compute_use_shares_3(
        id0_df, cd0_mat, i0_mat, g0_mat, nm0_rgm, md0_rmg)

    Z  = build_Z_3(dd0_mat, dm0_mat, T_dir, T_mar, d_int, m_int, sectors, n, S)
    F  = build_F_3(dd0_mat, dm0_mat, T_dir, T_mar, dC, dI, dG, mC, mI, mG, sectors, n, S)
    VA = (ld0_mat + kd0_mat).reshape(n * S)
    EX = x0_mat.reshape(n * S)

    # reference output + taxes kept SEPARATE by type (OECD basic-price schema)
    ys0_mat = (params['ys0_'][params['ys0_']['yr'] == year].groupby(['r', 's'])['value'].sum()
               .unstack('s').reindex(index=regions, columns=sectors, fill_value=0.0)
               .fillna(0.0).values)
    ty0_mat = load_matrix('ty0_', year, regions, sectors)
    tm0_mat = load_matrix('tm0_', year, regions, sectors)
    ta0_mat = load_matrix('ta0_', year, regions, sectors)

    M        = m0_mat.reshape(n * S)                                    # total imports
    M_interm = (m0_mat[:, :, None] * d_int).sum(axis=1).reshape(n * S)  # intermediate imports
    # FD split of ROW imports (direct shares) — v2-schema bookkeeping block.
    # For non-margin goods M_interm + M_fd_C + M_fd_I + M_fd_G == M (shares sum to 1);
    # margin goods route 100% to intermediates via the d_int fallback.
    M_fd_C   = (m0_mat * dC).reshape(n * S)
    M_fd_I   = (m0_mat * dI).reshape(n * S)
    M_fd_G   = (m0_mat * dG).reshape(n * S)

    interm_use = Z.reshape(n, S, n, S).sum(axis=(0, 3)).T  # (r, good) total intermediate use
    tax_prod = ty0_mat * ys0_mat                            # output tax       (r, sector)
    tariff   = tm0_mat * m0_mat                             # import duties    (r, good)
    tls_int  = ta0_mat * interm_use                         # product tax on intermediate (r, good)
    tls_fd   = ta0_mat * (cd0_mat + i0_mat + g0_mat)        # product tax on final use     (r, good)
    taxes    = (tax_prod + tariff + tls_int + tls_fd).reshape(n * S)

    n_ok = sum((T_dir[g].sum()+T_mar[g].sum() > 0 for g in sectors))
    print(f'  {year}: reconstructed {n_ok}/{S} sectors | Z sum={Z.sum():.1f} F sum={F.sum():.1f}')
    return dict(
        Z=Z, F=F, VA=VA, EX=EX, M=M, M_interm=M_interm, taxes=taxes,
        Y=ys0_mat.reshape(n * S),
        M_fd_C=M_fd_C, M_fd_I=M_fd_I, M_fd_G=M_fd_G,
        tax_prod=tax_prod.reshape(n * S), tariff=tariff.reshape(n * S),
        tls_int=tls_int.reshape(n * S),   tls_fd=tls_fd.reshape(n * S),
        ta0=ta0_mat, tm0=tm0_mat, ty0=ty0_mat,
        regions=np.array(regions), sectors=np.array(sectors))


In [ ]:
def save_table(table, version, year):
    """Save one year's build under grav_fric_<version>/IOT_<year>.npz.

    Also (re)writes a README.md in the build folder documenting this build: the
    free-text COMMENT, the list of YEARS and the gamma values (all set together
    in the config cell above).
    """
    path = build_path(version, year)
    np.savez_compressed(path, **table)

    readme = build_dir(version) / 'README.md'
    readme.write_text(
        f"# Build grav_fric_{version}\n\n"
        f"{COMMENT}\n\n"
        f"- Years: {', '.join(years)}\n"
        f"- gamma_trade: {GAMMA_TRADE}\n"
        f"- gamma_margin: {GAMMA_MARGIN}\n"
    )

    print(f"saved {version} {year} -> {path}   Z{table['Z'].shape}  sumZ={table['Z'].sum():.1f}")
    return path

### Save (grav_fric schema, version-tagged)

Builds every requested year (single year or list) and saves each one under
`grav_fric_<VERSION>/IOT_<year>.npz`. Bump `VERSION` in the config cell to write to
a fresh directory instead of overwriting a previous build.

In [ ]:
# ── Build + save every requested year (single year or list, no rigid overwrite) ─
builds = {year: build_table(year) for year in years}
for year, table in builds.items():
    save_table(table, VERSION, year)

next todo:
Why is M_interm created? why do we multiply it by d_int but the rest of the imports is not rerouted toward the final consumption?

### Diagnostic

In [ ]:
# ── Load everything the diagnostic needs (re-runnable) ───────────────
# Prereqs: run Setup (GDX `params`, regions, sectors, n, S, load_matrix) and the
# Construction config cell (paths + VERSION). The heavy build is NOT needed: the
# table is reloaded from disk, so you can jump here after Setup + config.
DIAG_VERSION = VERSION      # which build to inspect (defaults to the one just built)
DIAG_YEAR    = '2017' if '2017' in years else years[0]     # diagnostic / analysis operate on a SINGLE year
YEAR         = DIAG_YEAR    # alias consumed by the Taxes cell below

data_v3 = np.load(build_path(DIAG_VERSION, DIAG_YEAR), allow_pickle=True)
Z, F   = data_v3['Z'], data_v3['F']
VA, EX = data_v3['VA'], data_v3['EX']
M, M_interm, taxes = data_v3['M'], data_v3['M_interm'], data_v3['taxes']   # SAVED taxes/imports

# WiNDC reference (gross output + supply components) straight from the GDX
# ys0_ has both 's' and 'g' dims -> aggregate by SECTOR s (load_matrix would pick 'g')
ys0_mat = (params['ys0_'][params['ys0_']['yr'] == DIAG_YEAR].groupby(['r', 's'])['value'].sum()
           .unstack('s').reindex(index=regions, columns=sectors, fill_value=0.0)
           .fillna(0.0).values)
xd0_mat = load_matrix('xd0_', DIAG_YEAR, regions, sectors)
xn0_mat = load_matrix('xn0_', DIAG_YEAR, regions, sectors)
x0_mat  = load_matrix('x0_',  DIAG_YEAR, regions, sectors)

print(f"loaded {DIAG_VERSION} {DIAG_YEAR}: Z{Z.shape} sumZ={Z.sum():.1f} | F {F.sum():.1f} | "
      f"VA {VA.sum():.1f} | EX {EX.sum():.1f} | M {M.sum():.1f} | "
      f"M_interm {M_interm.sum():.1f} | taxes {taxes.sum():.1f}")

In [ ]:
# ── DIAGNOSTIC: reference-equilibrium identities (uses the SAVED v3 table) ─────
Y  = ys0_mat.reshape(n * S)                          # gross output (WiNDC reference)
s0 = (xd0_mat + xn0_mat + x0_mat)                    # total supply (reference)

# Row (supply) identity:  row(Z) + row(F) + EX = xd0 + xn0 + x0
row_sum = (Z.sum(axis=1) + F.sum(axis=1) + EX).reshape(n, S)
ras_err_mat = s0 - row_sum

# Column (cost) identity vs gross output:  ys0 = colZ + M_interm + VA + taxes + residual
col_Z    = Z.sum(axis=0)
residual = Y - (col_Z + M_interm + VA + taxes)       # SAVED M_interm & taxes
mask = Y > 0.1

# Row = Column:  total sales (row) = total production (col)
Y_row_flat = Z.sum(axis=1) + F.sum(axis=1) + EX
Y_col_flat = Z.sum(axis=0) + M_interm + VA + taxes   # SAVED M_interm & taxes
rc_gap = Y_row_flat - Y_col_flat
m_rc   = Y_row_flat > 0.1

print('=== Row identity:  row(Z)+row(F)+EX = xd0+xn0+x0 ===')
print(f'  S row_sum {row_sum.sum():.1f}  vs  S s0 {s0.sum():.1f}  | gap {ras_err_mat.sum():+.2f} '
      f'({ras_err_mat.sum()/s0.sum()*100:+.3f}%)  | S|err|/Ss0 {np.abs(ras_err_mat).sum()/s0.sum()*100:.3f}%')
print('=== Column identity:  ys0 = colZ + M_interm + VA + taxes + residual ===')
print(f'  S check {(col_Z+M_interm+VA+taxes).sum():.1f}  vs  S Y {Y.sum():.1f}  | resid {residual.sum():+.1f} '
      f'({residual.sum()/Y.sum()*100:+.2f}%)  | S|err|/SY {np.abs(residual[mask]).sum()/Y[mask].sum()*100:.2f}%')
print('=== Row = Column:  total sales (row) = total production (col) ===')
print(f'  S Y_row {Y_row_flat.sum():.1f}  vs  S Y_col {Y_col_flat.sum():.1f}  | gap {rc_gap.sum():+.1f} '
      f'({rc_gap.sum()/Y_row_flat.sum()*100:+.2f}%)  | S|err|/SY_row {np.abs(rc_gap[m_rc]).sum()/Y_row_flat[m_rc].sum()*100:.2f}%')

### Analyse

In [ ]:
# ── ANALYSE: residual by sector — 6 panels (3 identities x {% of output, $bn}) v3 ──
# Uses residual (column), ras_err_mat (row) and rc_gap (col=row) from the Diagnostic cell.
import matplotlib.pyplot as plt
import pandas as pd

out_sec   = Y.reshape(n, S).sum(axis=0)             # gross output per sector ($bn)
out_share = out_sec / out_sec.sum() * 100
out_safe  = np.where(out_sec > 0.1, out_sec, np.nan)

def _abs_by_sector(flat):                            # Sum_r|err| per sector ($bn)
    return np.abs(flat.reshape(n, S)).sum(axis=0)

PANELS = [
    ('Column identity  |ys0 - (colZ+M_interm+VA+taxes)|', residual),
    ('Row identity  |s0 - (rowZ+rowF+EX)|',               ras_err_mat.reshape(n * S)),
    ('Col = Row  |Y_row - Y_col|  (sales vs production)', rc_gap),
]

fig, axes = plt.subplots(3, 2, figsize=(19, 15))
rows = []
for (title, vec), (axL, axR) in zip(PANELS, axes):
    res_abs = _abs_by_sector(vec)                                            # $bn
    res_pct = np.where(out_sec > 0.1, res_abs / out_safe * 100, 0.0)         # % of output
    order = np.argsort(out_sec)[::-1] 
    x = np.arange(S)

    # LEFT: residual as % of OUTPUT + output-share overlay (secondary axis)
    axL.bar(x, res_pct[order], color=['#d62728' if e > 5 else '#1f77b4' for e in res_pct[order]])
    axL.axhline(5, color='orange', ls='--', lw=0.8)
    axL.set_xticks(x); axL.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axL.set_ylabel('residual (% of output)'); axL.set_title(f'{title}  -  % of output', fontsize=9)
    a2 = axL.twinx(); a2.plot(x, out_share[order], 'k.-', ms=4, lw=1.1, label='output share')
    a2.set_ylabel('output share (%)'); a2.set_ylim(0, out_share.max() * 1.15); a2.legend(loc='upper center', fontsize=8)

    # RIGHT: output & residual in $bn (residual bars + output line on secondary axis)
    axR.bar(x, res_abs[order], color='#1f77b4', label='residual')
    axR.set_xticks(x); axR.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axR.set_ylabel('residual ($bn)'); axR.set_title(f'{title}  -  output & residual ($bn)', fontsize=9)
    axR.legend(loc='upper right', fontsize=8)
    a3 = axR.twinx(); a3.plot(x, out_sec[order], 'k.-', ms=4, lw=1.1, label='output ($bn)')
    a3.set_ylabel('output ($bn)'); a3.legend(loc='upper center', fontsize=8)

    v = res_pct[out_sec > 0.1]
    rows.append({'identity': title.split('  ')[0],
                 'resid $bn': round(res_abs.sum(), 1), 'output $bn': round(out_sec.sum(), 1),
                 'resid %out (wtd)': round(res_abs.sum() / out_sec.sum() * 100, 2),
                 'mean%': round(v.mean(), 2), 'median%': round(np.median(v), 2),
                 'p95%': round(np.percentile(v, 95), 2), 'max%': round(v.max(), 2)})
plt.tight_layout(); plt.show()

print('v3 per-sector residual summary ($ and % of output):')
print(pd.DataFrame(rows).to_string(index=False))

**Important point:**

col is not supposed to equal row, because on the row we have the supply of good g (row in commodity or good) whereas on the column we have the production of sector s (column in industry or sector s). the right check is Σ_s ys0[s,g] = supply of good g, Σ_g ys0[s,g] = output of sector s. It is consistent by WiNDC construction. The "10%" is then not an error, just the byproducts effect: an industry can produce several goods / a good can be produced by several industries.

The only moment where the condition holds is when the table is **symmetric**: industry×industry or good×good. This requires either
1. Industry-technology assumption (the most used, no negatives): we reallocate via the make shares. Result: square good=sector table on both axes → row=col holds by construction, without RAS nor arbitrary distortion.
2. Commodity-technology (can produce negatives).

WiNDC provides ys0[s,g], so it is directly feasible. It is the standard method in IO analysis to obtain a balanced symmetric table.

## RAS normalization (column-target, rows preserved)

In [ ]:
# ── RAS: balance columns onto ys0 while PRESERVING row sums ─────────────
# Double-RAS (rows<->cols). Row target = CURRENT row sums -> row identity
# row(Z)+rowF+EX = s0 stays unchanged. Column target = ys0 - imports - VA - taxes
# -> column identity colZ+M_interm+VA+taxes = ys0 improves. col=row stays = the
# structural make/byproducts asymmetry (supply of good != output of sector).
def ras_2(Z0, u, v, max_iter=300, tol=1e-6):
    Z = Z0.copy().astype(float)
    for _ in range(max_iter):
        rs = Z.sum(1); Z *= np.divide(u, rs, out=np.ones_like(u), where=rs > 1e-12)[:, None]
        cs = Z.sum(0); Z *= np.divide(v, cs, out=np.ones_like(v), where=cs > 1e-12)[None, :]
        if max(np.abs(Z.sum(1) - u).max(), np.abs(Z.sum(0) - v).max()) < tol:
            break
    return Z

u_tgt = Z.sum(axis=1)                                   # PRESERVE current row sums
v_tgt = (Y - (M_interm + VA + taxes)).clip(0)           # column target: colZ = ys0 - imports - VA - taxes
v_tgt = v_tgt * u_tgt.sum() / max(v_tgt.sum(), 1e-10)   # Sum v = Sum u (RAS feasibility)
Z_ras = ras_2(Z, u_tgt, v_tgt)

s0f = s0.reshape(n * S)
def _ids(Zx):
    yr = Zx.sum(1) + F.sum(1) + EX
    yc = Zx.sum(0) + M_interm + VA + taxes
    m = Y > 0.1
    return (np.abs(Y - yc)[m].sum() / Y[m].sum() * 100,
            np.abs(s0f - yr)[m].sum() / s0f[m].sum() * 100,
            np.abs(yr - yc)[m].sum() / yr[m].sum() * 100)
print(f'{"":7} {"column":>8} {"row(vs s0)":>11} {"col=row":>9}')
for lbl, Zx in [('v3', Z), ('v3_RAS', Z_ras)]:
    c, r, rc = _ids(Zx)
    print(f'{lbl:7} {c:7.2f}% {r:10.2f}% {rc:8.2f}%')

# NOTE (fix): this cell used to save a TRUNCATED npz here (Z/F/VA/EX/M/M_interm/
# taxes only), dropping the tax blocks (tax_prod, tariff, tls_int, tls_fd,
# ta0/tm0/ty0). Those truncated files made grav_fric_<v>_RAS schema-inconsistent
# and apply_va_convention could not run on them. The single writer of the _RAS
# build is now the multi-year loop below (ras_one_year), which carries over
# EVERY block. This cell is diagnostic-only.


## Analyse — v3 vs v3_RAS

In [ ]:
# ── ANALYSE: residual by sector — v3 vs v3_RAS (6 panels) ──────────────────────
import matplotlib.pyplot as plt
import pandas as pd

out_sec   = Y.reshape(n, S).sum(axis=0)
out_share = out_sec / out_sec.sum() * 100
out_safe  = np.where(out_sec > 0.1, out_sec, np.nan)
def _abs_by_sector(flat): return np.abs(flat.reshape(n, S)).sum(axis=0)

def r_col(Zx, Fx): return Y      - (Zx.sum(0) + M_interm + VA + taxes)
def r_row(Zx, Fx): return s0f    - (Zx.sum(1) + Fx.sum(1) + EX)
def r_rc (Zx, Fx): return (Zx.sum(1) + Fx.sum(1) + EX) - (Zx.sum(0) + M_interm + VA + taxes)

models = {'v3': (Z, F), 'v3_RAS': (Z_ras, F)}
colors = {'v3': '#1f77b4', 'v3_RAS': '#2ca02c'}
PANELS = [('Column identity', 'col', r_col), ('Row identity', 'row', r_row), ('Col = Row', 'rc', r_rc)]

fig, axes = plt.subplots(3, 2, figsize=(20, 16))
rows = []
for (title, tag, fn), (axL, axR) in zip(PANELS, axes):
    res_abs = {nm: _abs_by_sector(fn(Zx, Fx)) for nm, (Zx, Fx) in models.items()}
    res_pct = {nm: np.where(out_sec > 0.1, res_abs[nm] / out_safe * 100, 0.0) for nm in models}
    order = np.argsort(out_sec)[::-1]
    x = np.arange(S)
    for i, nm in enumerate(models):
        axL.bar(x + (i - 0.5) * 0.4, res_pct[nm][order], width=0.4, color=colors[nm], alpha=0.85, label=nm)
    axL.axhline(5, color='orange', ls='--', lw=0.8)
    axL.set_xticks(x); axL.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axL.set_ylabel('residual (% of output)'); axL.set_title(f'{title}  -  % of output', fontsize=9); axL.legend(loc='upper right', fontsize=8)
    a2 = axL.twinx(); a2.plot(x, out_share[order], 'k.-', ms=4, lw=1.1, label='output share')
    a2.set_ylabel('output share (%)'); a2.set_ylim(0, out_share.max() * 1.15); a2.legend(loc='upper center', fontsize=8)
    for i, nm in enumerate(models):
        axR.bar(x + (i - 0.5) * 0.4, res_abs[nm][order], width=0.4, color=colors[nm], alpha=0.85, label=nm)
    axR.set_xticks(x); axR.set_xticklabels([sectors[j] for j in order], rotation=90, fontsize=6)
    axR.set_ylabel('residual ($bn)'); axR.set_title(f'{title}  -  output & residual ($bn)', fontsize=9); axR.legend(loc='upper right', fontsize=8)
    a3 = axR.twinx(); a3.plot(x, out_sec[order], 'k.-', ms=4, lw=1.1, label='output ($bn)')
    a3.set_ylabel('output ($bn)'); a3.legend(loc='upper center', fontsize=8)
    for nm in models:
        v = res_pct[nm][out_sec > 0.1]
        rows.append({'identity': tag, 'model': nm, 'resid $bn': round(res_abs[nm].sum(), 1),
                     'resid %out (wtd)': round(res_abs[nm].sum() / out_sec.sum() * 100, 2),
                     'mean%': round(v.mean(), 2), 'median%': round(np.median(v), 2),
                     'p95%': round(np.percentile(v, 95), 2), 'max%': round(v.max(), 2)})
plt.tight_layout(); plt.show()
print('v3 vs v3_RAS per-sector residual summary:')
print(pd.DataFrame(rows).to_string(index=False))

Conclusion: the RAS fitting clearly improves the results.

In [ ]:
# ── Apply the column-target RAS to EVERY year and save each one ───────────────
# Generalises the single-year RAS above (DIAG_YEAR) to the full `years` list.
# Reuses ras_2 (defined above). For each year: reload the saved v3 table, RAS Z
# onto ys0 while preserving row sums, and write the result under
# grav_fric_<DIAG_VERSION>_RAS/IOT_<year>.npz.
RAS_VERSION = f'{DIAG_VERSION}_RAS'

def ras_one_year(year):
    """RAS one year's saved v3 table onto ys0; save under RAS_VERSION. Returns
    the column-identity residual (% of gross output) before and after RAS."""
    d   = np.load(build_path(DIAG_VERSION, year), allow_pickle=True)
    out = {k: d[k] for k in d.files}         # carry over EVERY block (tax_prod, tariff, ta0, ...)
    Zy, VAy, Mi, txy = out['Z'], out['VA'], out['M_interm'], out['taxes']

    # gross-output target ys0 for this year (aggregate over industry s)
    ys0 = (params['ys0_'][params['ys0_']['yr'] == year].groupby(['r', 's'])['value'].sum()
           .unstack('s').reindex(index=regions, columns=sectors, fill_value=0.0)
           .fillna(0.0).values).reshape(n * S)

    u = Zy.sum(axis=1)                       # PRESERVE current row sums
    v = (ys0 - (Mi + VAy + txy)).clip(0)     # column target: colZ = ys0 - imports - VA - taxes
    v = v * u.sum() / max(v.sum(), 1e-10)    # feasibility: sum v = sum u
    Z_ras = ras_2(Zy, u, v)

    out['Z'] = Z_ras                         # Z is rebalanced
    # tls_int (product tax on intermediate use) is the only Z-dependent tax block:
    # recompute it from the RAS'd Z so every saved amount stays = rate x its actual
    # base, then refresh the combined taxes row. (tax_prod/tariff/tls_fd bases -
    # ys0, imports, final demand - are untouched by RAS, so they stay coherent.)
    interm_use     = Z_ras.reshape(n, S, n, S).sum(axis=(0, 3)).T        # (r, good)
    out['tls_int'] = (out['ta0'] * interm_use).reshape(n * S)
    out['taxes']   = out['tax_prod'] + out['tariff'] + out['tls_int'] + out['tls_fd']
    np.savez_compressed(build_path(RAS_VERSION, year), **out)

    mask = ys0 > 0.1
    col_res = lambda Zx: np.abs(ys0 - (Zx.sum(0) + Mi + VAy + txy))[mask].sum() / ys0[mask].sum() * 100
    return col_res(Zy), col_res(Z_ras)

print(f'RAS all years -> grav_fric_{RAS_VERSION}')
print(f'{"year":>6}{"col resid v3":>15}{"col resid RAS":>16}')
for year in years:
    before, after = ras_one_year(year)
    print(f'{year:>6}{before:13.2f} %{after:14.2f} %')
print(f'done: {len(years)} years saved to grav_fric_{RAS_VERSION}/')

## Taxes

### Tax analysis

In [ ]:
# ── TAXES: kept SEPARATE by type (rates + amounts recomputed from the table) ───
# OECD ICIO keeps product taxes OUT of Z (basic prices) as a separate 'TLS' row.
# WiNDC rates are structural (survive RAS); amounts are recomputed from current Z.
# Analysis is run on the POST-RAS table -> reload Z/F/VA/EX/M/... from the _RAS build.
RAS_VERSION = f'{DIAG_VERSION}_RAS'
_d = np.load(build_path(RAS_VERSION, YEAR), allow_pickle=True)
Z, F, VA, EX       = _d['Z'], _d['F'], _d['VA'], _d['EX']
M, M_interm, taxes = _d['M'], _d['M_interm'], _d['taxes']
print(f'taxes/analysis on RAS table -> grav_fric_{RAS_VERSION}/IOT_{YEAR}.npz')

def _rate(p, dim=None):
    df = params[p]; df = df[df['yr'] == YEAR]
    d = 'g' if 'g' in df.columns else 's'        # auto-detect (ta0/tm0 by good, ty0/cd0/i0/g0 by sector)
    return (df.groupby(['r', d])['value'].sum().unstack(d)
            .reindex(index=regions, columns=sectors, fill_value=0.0).fillna(0.0).values)
ta0_mat = _rate('ta0_', 'g')      # absorption (product) tax rate, by (r, good)
tm0_mat = _rate('tm0_', 'g')      # import tariff rate,           by (r, good)
ty0_mat = _rate('ty0_', 's')      # output (production) tax rate, by (r, sector)

m0_mat = M.reshape(n, S)                                   # imports by (r, good)
cd0 = _rate('cd0_', 'g'); i0 = _rate('i0_', 'g'); g0 = _rate('g0_', 'g')
fd_tot = cd0 + i0 + g0                                      # final absorption by (r, good)

# Intermediate-use base comes from the (possibly RAS'd) Z -> recompute.
Z4 = Z.reshape(n, S, n, S)                                  # [origin_r, good, dest_r, sector]
interm_use = Z4.sum(axis=(0, 3)).T                          # (dest_r, good): use of each good as intermediate

# Monetary AMOUNTS, each kept separate (decide placement later):
tax_prod = ty0_mat * ys0_mat            # D29-like, output tax       (r, sector)
tariff   = tm0_mat * m0_mat             # import duties              (r, good)
tls_int  = ta0_mat * interm_use         # product tax on intermediate (r, good)  [recomputed from Z]
tls_fd   = ta0_mat * fd_tot             # product tax on final use   (r, good)

print('Tax blocks (separate) -- 2017, $bn:')
print(f'  tax_prod (ty0*ys0)            sum={tax_prod.sum():8.1f}   rate by (r,sector)')
print(f'  tariff   (tm0*m0)             sum={tariff.sum():8.1f}   rate by (r,good)')
print(f'  tls_int  (ta0*interm. use)    sum={tls_int.sum():8.1f}   recomputed from Z')
print(f'  tls_fd   (ta0*final demand)   sum={tls_fd.sum():8.1f}')
print(f'  --> TLS (products) = tls_int+tls_fd = {(tls_int+tls_fd).sum():.1f}   (OECD taxes-less-subsidies on products)')

### Visualization — table blocks (NY)

In [ ]:
# ── VISUALIZATION: how the separate blocks articulate (focus on NY) ───────────
import matplotlib.pyplot as plt

ny  = regions.index('NY')
Z4  = Z.reshape(n, S, n, S)
F4  = F.reshape(n, S, n, 3)
x   = np.arange(S)
nysl = slice(ny * S, ny * S + S)

fig = plt.figure(figsize=(20, 12), dpi=200)

# 1) NY -> NY intermediate block (sector x sector)
ax = plt.subplot(2, 3, 1)
im = ax.imshow(np.log1p(Z4[ny, :, ny, :]), cmap='viridis', aspect='auto')
ax.set_title('NY -> NY intermediate Z  (log1p)'); ax.set_xlabel('buying sector'); ax.set_ylabel('selling sector')
ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=4); ax.set_yticks(x); ax.set_yticklabels(sectors, fontsize=4)
plt.colorbar(im, ax=ax, shrink=0.7)

# 2) Interstate Z structure (region x region, summed over sectors)
ax = plt.subplot(2, 3, 2)
ZRR = Z4.sum(axis=(1, 3))
im = ax.imshow(np.log1p(ZRR), cmap='magma', aspect='auto')
ax.set_title('Interstate Z (sum over sectors, log1p)'); ax.set_xlabel('dest state'); ax.set_ylabel('origin state')
ax.set_xticks(range(n)); ax.set_xticklabels(regions, rotation=90, fontsize=4); ax.set_yticks(range(n)); ax.set_yticklabels(regions, fontsize=4)
plt.colorbar(im, ax=ax, shrink=0.7)

# 3) NY column = cost composition by sector (Z inputs + imports + VA + taxes)
ax = plt.subplot(2, 3, 3)
colZ = Z.sum(0)[nysl]; Mi = M_interm[nysl]; va = VA[nysl]; tx = taxes[nysl]
ax.bar(x, colZ, label='interm Z',  color='#1f77b4')
ax.bar(x, Mi, bottom=colZ, label='imports', color='#9467bd')
ax.bar(x, va, bottom=colZ + Mi, label='VA', color='#2ca02c')
ax.bar(x, tx, bottom=colZ + Mi + va, label='taxes', color='#d62728')
ax.set_title('NY: production cost composition'); ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=4); ax.legend(fontsize=7)

# 4) NY final demand by good (C/I/G)
ax = plt.subplot(2, 3, 4)
Fny = F4[:, :, ny, :].sum(0)
ax.bar(x, Fny[:, 0], label='C', color='#1f77b4')
ax.bar(x, Fny[:, 1], bottom=Fny[:, 0], label='I', color='#ff7f0e')
ax.bar(x, Fny[:, 2], bottom=Fny[:, 0] + Fny[:, 1], label='G', color='#2ca02c')
ax.set_title('NY final demand by good'); ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=4); ax.legend(fontsize=7)

# 5) Block structure summary (separate tables)
ax = plt.subplot(2, 3, 5); ax.axis('off')
txt = ('BLOCK STRUCTURE (separate tables, $bn)\n\n'
       f'Z      {Z.shape}   sum={Z.sum():8.0f}\n'
       f'F      {F.shape}   sum={F.sum():8.0f}\n'
       f'VA     ({n*S},)      sum={VA.sum():8.0f}\n'
       f'EX     ({n*S},)      sum={EX.sum():8.0f}\n'
       f'M      ({n*S},)      sum={M.sum():8.0f}\n'
       f'M_interm            sum={M_interm.sum():8.0f}\n\n'
       f'taxes (combined)    sum={taxes.sum():8.0f}\n'
       f'  tax_prod={tax_prod.sum():7.0f}  tariff={tariff.sum():6.0f}\n'
       f'  tls_int ={tls_int.sum():7.0f}  tls_fd={tls_fd.sum():6.0f}')
ax.text(0.0, 0.95, txt, fontsize=10, va='top', family='monospace')

# 6) Tax blocks by type (national, by sector/good)
# 6) Tax blocks by type (national, by sector/good)
ax = plt.subplot(2, 3, 6)
tax_blocks = [('tax_prod (ty0)', tax_prod.sum(0), '#1f77b4'),
              ('tariff (tm0)',   tariff.sum(0),   '#ff7f0e'),
              ('tls_int (ta0)',  tls_int.sum(0),  '#2ca02c'),
              ('tls_fd (ta0)',   tls_fd.sum(0),   '#d62728')]
pos = np.zeros(S); neg = np.zeros(S)                       # separate stacks for +/-
for label, vals, color in tax_blocks:
    p = np.clip(vals, 0, None); m = np.clip(vals, None, 0)
    ax.bar(x, p, bottom=pos, color=color, label=label)     # positive part stacks up
    ax.bar(x, m, bottom=neg, color=color)                  # negative part stacks down
    pos += p; neg += m
ax.axhline(0, color='k', lw=0.6)
ax.set_title('Tax blocks by sector/good (national)'); ax.set_xticks(x); ax.set_xticklabels(sectors, rotation=90, fontsize=4); ax.legend(fontsize=7)

plt.tight_layout(); plt.show()

### VA convention (OECD / SNA): production tax → gross VA

**Decision (enacted).** Following the OECD ICIO / SNA basic-price convention, the WiNDC
tax blocks are placed as:

- **TLS** (Taxes Less Subsidies on *products*, D21−D31) = `tls_int + tls_fd + tariff`
  — taxes on the use of goods (`ta0`-based, **net of subsidies**: `ta0`<0 = subsidy) + import duties (`tm0`).
- **Gross VA** = `ld0 + kd0 + tax_prod` — labour + capital + *other taxes on production*
  (D29, the WiNDC output tax `ty0`).

**Why this choice.** Validated empirically (`analysis_comparison`): moving `tax_prod` into VA
leaves the VA↔OECD-VA correlation unchanged (pearson **0.998**) and makes the magnitude match
**exactly** (ratio **0.96 → 1.00**). The remaining TLS block stays a genuine WiNDC↔OECD
methodological difference in how product taxes are distributed by sector (not a placement issue).

It is a **relabeling**: the column total (gross output `ys0`) is unchanged, since
`VA + TLS = (ld0+kd0) + (tax_prod+tariff+tls_int+tls_fd) = old VA + old taxes`.

In [ ]:
# ── Enact the VA convention on EVERY post-RAS year (idempotent) ───────────────
def apply_va_convention(d):
    """OECD/SNA: move the production tax (tax_prod) into gross VA; set the tax row to
    TLS (taxes less subsidies on products = tls_int + tls_fd + tariff).
    Relabeling only -> the column total (output) is unchanged. Idempotent."""
    keys = d.files if hasattr(d, 'files') else d.keys()
    d = {k: d[k] for k in keys}
    if 'va_convention' in d:                                   # already applied -> no-op
        return d
    d['VA']    = d['VA'] + d['tax_prod']                       # gross VA incl. other production tax (D29)
    d['TLS']   = d['tariff'] + d['tls_int'] + d['tls_fd']      # taxes less subsidies on products
    d['taxes'] = d['TLS'].copy()                               # the tax row is now TLS only
    d['va_convention'] = np.array(True)
    return d

# Apply to ALL years of the post-RAS build (previously only YEAR was relabeled,
# which left grav_fric_<v>_RAS inconsistent across years on a full run).
VA_VERSION = f'{VERSION}_RAS'              # which build to relabel (post-RAS tables)
for year in years:
    f = build_path(VA_VERSION, year)
    if not f.exists():
        print(f'{f.name}: MISSING -> run the RAS-all-years cell first')
        continue
    d0 = np.load(f, allow_pickle=True)
    if 'tax_prod' not in d0.files and 'va_convention' not in d0.files:
        print(f'{f.name}: legacy truncated schema (no tax blocks) -> re-run ras_one_year for this year')
        continue
    va_b, tax_b = float(d0['VA'].sum()), float(d0['taxes'].sum())
    d1 = apply_va_convention(d0)
    np.savez_compressed(f, **d1)
    print(f"{f.name}: VA {va_b:7.0f} -> {d1['VA'].sum():7.0f}  |  "
          f"taxes {tax_b:6.0f} -> TLS {d1['taxes'].sum():6.0f}  "
          f"(va_convention={bool(d1['va_convention'])})")
